# 5.3 Boosting

- 将多个较弱的模型组合为一个较强的模型
- **降低模型的偏差**

按顺序训练 n 个弱模型，第 i 个模型：
- 评估现有模型的误差 $\boldsymbol{\epsilon_t}$
- Train a weak learner $\boldsymbol{\hat{f}_t}$, 关注上一个模型预测错误的样本
  - AdaBoost: 根据 $\boldsymbol{\epsilon_t}$ 重新加权采样
  - Gradient boosting: Train learner to predict$\boldsymbol{\epsilon_t}$
- **Additively** combining existing weak learners with$\boldsymbol{\hat{f}_t}$

### Gradient Boosting
- Supports arbitrary differentiable loss  
- $H_t(x)$: output of combined model at timestep $t$, with $H_1(x) = 0$
- For each step $t$, repeat:  
  - Train a new learner $\hat{f}_t$on residuals:$\{(x_i, y_i - H_t(x_i))\}_{i=1, \dots, m}$
  - Combine:$H_{t+1}(x) = H_t(x) + \eta \hat{f}_t(x)$ (shrinkage parameter $\eta$ for regularization，使用 $\eta=1$ 完全拟合残差容易过拟合)  

理解：
- MSE $L = \frac{1}{2}(H(x) - y)^2$, residual equals negative gradient $y - H(x) = -\frac{\partial L}{\partial H}$
- For other loss $L$, learner $\hat{f}_t = \arg\min \frac{1}{2} \left( \hat{f}_t(x) + \frac{\partial L(x)}{\partial H_t} \right)^2$  
- Avoid overfitting: subsampling, shrinkage, early-stopping

In [ ]:
class GradientBoosting:
    def __init__(self, base_learner, n_learners, learning_rate):
        self.learners = [clone(base_learner) for _ in range(n_learners)]
        self.lr = learning_rate

    def fit(self, X, y):
        residual = y.copy()
        for learner in self.learners:
            learner.fit(X, residual)
            residual -= self.lr * learner.predict(X)

    def predict(self, X):
        preds = [learner.predict(X) for learner in self.learners]
        return np.array(preds).sum(axis=0) * self.lr

### Gradient Boosting Decision Trees (GBDT)
决策树是一个强模型，需要正则化变为弱模型：限制层数 + 随机采样特征

相比于随机森林，GBDT没有多少过拟合（弱模型和学习率选择恰当时）

<img src="paste_image/2026-01-29-16-54-10.png" width="98%">

顺序化构建决策树太慢，一般使用加速算法：
- XGBoost（eXtreme Gradient Boosting，极端梯度提升）
- lightGBM（light Gradient Boosting Machine，轻量级梯度提升机）